# VTC-AUDIT — Reproduce all paper tables and key results

Author: Isaac(Rucheng) Li

The release already includes all six completed runs. Run this notebook directly to reconstruct the paper tables from those saved outputs; rerunning the three model notebooks is optional. This notebook does not load a VLM or run inference. It reads `VTC_AUDIT/results/run` and writes tables and verification files to `VTC_AUDIT/results/meta results`.

Generated outputs include Tables 1--6, the seed-2026 replication summaries, eligible-cell counts, and the integrity report.


In [ ]:
%pip -q install --upgrade-strategy only-if-needed "pandas>=2" "numpy>=1.26" "pyarrow>=14" "scikit-learn>=1.4,<2" "tabulate>=0.9"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("/content/drive/MyDrive/VTC_AUDIT")
RUN_ROOT = PROJECT_ROOT / "results" / "run"
META_ROOT = PROJECT_ROOT / "results" / "meta results"
META_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_SEED = 42
REPLICATION_SEED = 2026
SEEDS = [PRIMARY_SEED, REPLICATION_SEED]
PROTOCOL_TAG = "vtc_audit_wacv2027_divprune_apet_r25_uidfps_v1"
BOOTSTRAP_N = 5000
PRACTICAL_AUROC = 0.65
EXPECTED_RESULTS_PER_RUN = 118000
EXPECTED_CASES_PER_RUN = 32000

MODELS = {
    "qwen35": ("Qwen3.5-9B", "qwen3_5_9b"),
    "ov2": ("OneVision-2-8B", "llava_onevision_2_8b"),
    "internvl35": ("InternVL3.5-8B", "internvl3_5_8b"),
}
MODEL_ORDER = ["qwen35", "ov2", "internvl35"]
METHOD_ORDER = ["divprune", "apet"]
TASK_ORDER = ["chartqa", "textvqa", "gqa", "pope"]
FAMILY_ORDER = ["jpeg", "noise", "blur", "brightness"]
BENCHMARKS_BOOTSTRAP = ["pope", "textvqa", "gqa", "chartqa"]

MODEL_LABEL = {k: v[0] for k, v in MODELS.items()}
METHOD_LABEL = {"divprune": "DivPrune", "apet": "ApET"}
TASK_LABEL = {"chartqa": "ChartQA", "textvqa": "TextVQA", "gqa": "GQA", "pope": "POPE"}
FAMILY_LABEL = {"jpeg": "JPEG", "noise": "Noise", "blur": "Blur", "brightness": "Bright."}

def write_csv(frame, name):
    path = META_ROOT / name
    frame.to_csv(path, index=False)
    print("saved", path)
    return path

def stable_seed(*parts) -> int:
    text = "|".join(map(str, parts))
    return int.from_bytes(hashlib.blake2b(text.encode(), digest_size=8).digest(), "little") % (2**32)

print("run root :", RUN_ROOT)
print("meta root:", META_ROOT)


## 1. Load and validate the six completed runs

In [ ]:
RUN_ROWS = []
SUMMARY_PARTS = []
MONITOR_PARTS = []
SEVERITY_PARTS = []
RESULTS42_PARTS = []
CASES42_PARTS = []

result_columns = [
    "model_kind", "bench", "uid", "method", "budget", "shift_key",
    "support_mode", "score", "full_credit"
]
case_columns = [
    "model_kind", "method", "budget", "bench", "uid", "cluster_id",
    "shift_family", "severity", "eligible", "dynamic_score", "frozen_score",
    "dynamic_failure", "frozen_failure"
]

for seed in SEEDS:
    shared_manifest = RUN_ROOT / f"seed_{seed}" / "_shared" / "manifest.csv"
    if not shared_manifest.exists():
        raise FileNotFoundError(shared_manifest)
    manifest = pd.read_csv(shared_manifest, keep_default_na=False)
    if len(manifest) != 2000 or manifest["uid"].duplicated().any():
        raise RuntimeError(f"Invalid shared manifest for seed {seed}")
    expected_counts = {"chartqa": 500, "gqa": 500, "pope": 500, "textvqa": 500}
    if manifest.groupby("bench").size().to_dict() != expected_counts:
        raise RuntimeError(f"Unexpected task counts in seed {seed} manifest")

    for model_kind in MODEL_ORDER:
        model_name, slug = MODELS[model_kind]
        root = RUN_ROOT / f"seed_{seed}" / slug
        complete = json.loads((root / "complete.json").read_text())
        design = json.loads((root / "design.json").read_text())
        preflight = json.loads((root / "preflight.json").read_text())

        checks = [
            complete.get("pass") is True,
            complete.get("protocol_tag") == PROTOCOL_TAG,
            int(complete.get("seed", -1)) == seed,
            complete.get("model_kind") == model_kind,
            int(complete.get("result_rows", -1)) == EXPECTED_RESULTS_PER_RUN,
            int(complete.get("case_rows", -1)) == EXPECTED_CASES_PER_RUN,
            design.get("protocol_tag") == PROTOCOL_TAG,
            int(design.get("resample_seed", -1)) == seed,
            preflight.get("pass") is True,
        ]
        if not all(checks):
            raise RuntimeError(f"Run validation failed: {root}")

        for file_name, sink in [
            ("summary.csv", SUMMARY_PARTS),
            ("monitor.csv", MONITOR_PARTS),
            ("severity_summary.csv", SEVERITY_PARTS),
        ]:
            f = pd.read_csv(root / "analysis" / file_name)
            f["seed"] = seed
            f["model_name"] = model_name
            sink.append(f)

        if seed == PRIMARY_SEED:
            results = pd.read_parquet(root / "results.parquet", columns=result_columns)
            cases = pd.read_parquet(root / "analysis" / "cases.parquet", columns=case_columns)
            if len(results) != EXPECTED_RESULTS_PER_RUN or len(cases) != EXPECTED_CASES_PER_RUN:
                raise RuntimeError(f"Primary parquet row mismatch: {root}")
            results["model_name"] = model_name
            cases["model_name"] = model_name
            RESULTS42_PARTS.append(results)
            CASES42_PARTS.append(cases)

        RUN_ROWS.append({
            "seed": seed, "model_kind": model_kind, "model_name": model_name,
            "result_rows": int(complete["result_rows"]),
            "case_rows": int(complete["case_rows"]),
            "preflight_pass": bool(preflight["pass"]),
            "position_policy": design["position_policy"],
        })

RUNS = pd.DataFrame(RUN_ROWS)
SUMMARY = pd.concat(SUMMARY_PARTS, ignore_index=True)
MONITOR = pd.concat(MONITOR_PARTS, ignore_index=True)
SEVERITY = pd.concat(SEVERITY_PARTS, ignore_index=True)
RESULTS42 = pd.concat(RESULTS42_PARTS, ignore_index=True)
CASES42 = pd.concat(CASES42_PARTS, ignore_index=True)

write_csv(RUNS, "run_summary.csv")
print(RUNS.to_string(index=False))
print("total inference rows:", int(RUNS.result_rows.sum()))


## 2. Table 1 — performance and eligibility

In [ ]:
def mean_score_credit(frame):
    return float(frame["score"].mean()), float(frame["full_credit"].mean())

def eligible_rate(model_kind, method, bench=None):
    g = CASES42[(CASES42.model_kind == model_kind) & (CASES42.method == method) & (CASES42.budget == "r25")]
    if bench is not None:
        g = g[g.bench == bench]
    return float(g["eligible"].astype(bool).mean())

rows = []
for mk in MODEL_ORDER:
    model_results = RESULTS42[RESULTS42.model_kind == mk]
    for task in ["all"] + TASK_ORDER:
        bench = None if task == "all" else task
        f = model_results if bench is None else model_results[model_results.bench == bench]

        full_clean = f[(f.method == "full") & (f.budget == "full") & (f.shift_key == "clean") & (f.support_mode == "full")]
        full_shift = f[(f.method == "full") & (f.budget == "full") & (f.shift_key != "clean") & (f.support_mode == "full")]
        fc_score, fc_full = mean_score_credit(full_clean)
        fs_score, fs_full = mean_score_credit(full_shift)

        row = {
            "Model": MODEL_LABEL[mk],
            "Task": "All" if task == "all" else TASK_LABEL[task],
            "Full clean score": 100 * fc_score,
            "Full clean sigma=1": 100 * fc_full,
            "Full shifted score": 100 * fs_score,
            "Full shifted sigma=1": 100 * fs_full,
        }
        for method in METHOD_ORDER:
            comp = f[(f.method == method) & (f.budget == "r25") & (f.shift_key == "clean") & (f.support_mode == "clean")]
            cs, cf = mean_score_credit(comp)
            prefix = METHOD_LABEL[method]
            row[f"{prefix} clean score"] = 100 * cs
            row[f"{prefix} clean sigma=1"] = 100 * cf
            row[f"{prefix} eligibility"] = 100 * eligible_rate(mk, method, bench)
        rows.append(row)

TABLE1 = pd.DataFrame(rows)
write_csv(TABLE1.round(2), "Table1_seed42.csv")
display(TABLE1.round(2))


## 3. Table 2 — primary intervention results

In [ ]:
primary = SUMMARY[(SUMMARY.seed == PRIMARY_SEED) & (SUMMARY.budget == "r25")].copy()
primary["model_order"] = primary.model_kind.map({k:i for i,k in enumerate(MODEL_ORDER)})
primary["method_order"] = primary.method.map({k:i for i,k in enumerate(METHOD_ORDER)})
primary = primary.sort_values(["model_order", "method_order"])

TABLE2 = pd.DataFrame({
    "Model": primary.model_kind.map(MODEL_LABEL),
    "Compressor": primary.method.map(METHOD_LABEL),
    "Dynamic SCFR": 100 * primary.dynamic_scfr,
    "Frozen SCFR": 100 * primary.frozen_scfr,
    "Counterfactual SCFR": 100 * primary.counterfactual_scfr,
    "Delta": 100 * primary.dynamic_minus_frozen,
    "CI low": 100 * primary.dynamic_minus_frozen_ci_low,
    "CI high": 100 * primary.dynamic_minus_frozen_ci_high,
    "Persistence": 100 * primary.counterfactual_persistence,
})
write_csv(TABLE2.round(2), "Table2_seed42.csv")
display(TABLE2.round(2))


## 4. Table 3 — partial-credit sensitivity

This cell reproduces the paired benchmark-stratified image-cluster bootstrap used for the partial-credit sensitivity analysis.

In [ ]:
def point_macro_loss(frame, score_col, bench_subset=None):
    work = frame if bench_subset is None else frame[frame.bench.isin(bench_subset)]
    vals = []
    for (_, _), g in work.groupby(["bench", "shift_family"], sort=True):
        eg = g[g.eligible.astype(bool)]
        vals.append(float((1.0 - eg[score_col].astype(float)).mean()) if len(eg) else np.nan)
    return float(np.nanmean(vals))

def cluster_loss_arrays(frame, bench, family, score_col, clusters):
    g = frame[(frame.bench == bench) & (frame.shift_family == family)].copy()
    eligible = g.eligible.astype(bool)
    g["_num"] = np.where(eligible, 1.0 - g[score_col].astype(float), 0.0)
    g["_den"] = eligible.astype(float)
    num = g.groupby("cluster_id")["_num"].sum().reindex(clusters, fill_value=0.0).to_numpy(float)
    den = g.groupby("cluster_id")["_den"].sum().reindex(clusters, fill_value=0.0).to_numpy(float)
    return num, den

def paired_bootstrap_macro_loss(frame, bench_subset=None, n_boot=5000, seed=0):
    benches = BENCHMARKS_BOOTSTRAP if bench_subset is None else list(bench_subset)
    work = frame[frame.bench.isin(benches)].copy()
    rng = np.random.default_rng(seed)
    clusters = {b: sorted(work.loc[work.bench == b, "cluster_id"].unique()) for b in benches}
    sample_indices = {b: rng.integers(0, len(clusters[b]), size=(n_boot, len(clusters[b]))) for b in benches}
    arm_reps = {}
    for score_col in ["dynamic_score", "frozen_score"]:
        cell_reps = []
        for bench in benches:
            for family in sorted(work.loc[work.bench == bench, "shift_family"].unique()):
                num, den = cluster_loss_arrays(work, bench, family, score_col, clusters[bench])
                idx = sample_indices[bench]
                sampled_num = num[idx].sum(axis=1)
                sampled_den = den[idx].sum(axis=1)
                reps = np.divide(sampled_num, sampled_den, out=np.full(n_boot, np.nan), where=sampled_den > 0)
                cell_reps.append(reps)
        stacked = np.stack(cell_reps, axis=1)
        arm_reps[score_col] = np.nanmean(stacked, axis=1)
    return arm_reps

rows = []
for mk in MODEL_ORDER:
    for method in METHOD_ORDER:
        frame = CASES42[(CASES42.model_kind == mk) & (CASES42.method == method) & (CASES42.budget == "r25")].copy()
        out = {"Model": MODEL_LABEL[mk], "Compressor": METHOD_LABEL[method]}
        for scope, benches, label in [("all_tasks", None, "All"), ("textvqa_only", ["textvqa"], "TextVQA")]:
            dyn = point_macro_loss(frame, "dynamic_score", benches)
            frz = point_macro_loss(frame, "frozen_score", benches)
            reps = paired_bootstrap_macro_loss(
                frame, benches, BOOTSTRAP_N,
                stable_seed("partial_credit", PRIMARY_SEED, mk, method, scope),
            )
            diff_reps = reps["dynamic_score"] - reps["frozen_score"]
            lo, hi = np.quantile(diff_reps[np.isfinite(diff_reps)], [0.025, 0.975])
            out[f"{label} dynamic"] = 100 * dyn
            out[f"{label} frozen"] = 100 * frz
            out[f"{label} Delta"] = 100 * (dyn - frz)
            out[f"{label} CI low"] = 100 * lo
            out[f"{label} CI high"] = 100 * hi
        rows.append(out)

TABLE3 = pd.DataFrame(rows)
write_csv(TABLE3.round(2), "Table3_seed42.csv")
display(TABLE3.round(2))


## 5. Table 4 — task-by-shift heterogeneity and eligible-cell counts

In [ ]:
cell_rows = []
for mk in MODEL_ORDER:
    for method in METHOD_ORDER:
        f = CASES42[(CASES42.model_kind == mk) & (CASES42.method == method) & (CASES42.budget == "r25")].copy()
        for bench in TASK_ORDER:
            for family in FAMILY_ORDER:
                g = f[(f.bench == bench) & (f.shift_family == family)]
                eg = g[g.eligible.astype(bool)]
                dyn = float(eg.dynamic_failure.astype(bool).mean())
                frz = float(eg.frozen_failure.astype(bool).mean())
                cell_rows.append({
                    "model_kind": mk, "method": method, "bench": bench, "shift_family": family,
                    "eligible_configurations": int(len(eg)),
                    "eligible_image_clusters": int(eg.cluster_id.nunique()),
                    "dynamic_minus_frozen": dyn - frz,
                })
CELLS = pd.DataFrame(cell_rows)
write_csv(CELLS, "Table4_cell_data_seed42.csv")

wide_rows = []
for mk in MODEL_ORDER:
    for bench in TASK_ORDER:
        row = {"Model": MODEL_LABEL[mk], "Task": TASK_LABEL[bench]}
        for method in METHOD_ORDER:
            for family in FAMILY_ORDER:
                value = CELLS[(CELLS.model_kind == mk) & (CELLS.method == method) & (CELLS.bench == bench) & (CELLS.shift_family == family)].dynamic_minus_frozen.iloc[0]
                row[f"{METHOD_LABEL[method]} {FAMILY_LABEL[family]}"] = 100 * value
        wide_rows.append(row)
TABLE4 = pd.DataFrame(wide_rows)
write_csv(TABLE4.round(2), "Table4_seed42.csv")

CELL_COUNTS = pd.DataFrame([{
    "n_cells": len(CELLS),
    "eligible_config_min": int(CELLS.eligible_configurations.min()),
    "eligible_config_median": float(CELLS.eligible_configurations.median()),
    "eligible_config_max": int(CELLS.eligible_configurations.max()),
    "eligible_cluster_min": int(CELLS.eligible_image_clusters.min()),
    "eligible_cluster_median": float(CELLS.eligible_image_clusters.median()),
    "eligible_cluster_max": int(CELLS.eligible_image_clusters.max()),
}])
write_csv(CELL_COUNTS, "Table4_eligible_cell_summary_seed42.csv")
display(TABLE4.round(2))
display(CELL_COUNTS)


## 6. Table 5 — shift severity

In [ ]:
sev = SEVERITY[(SEVERITY.seed == PRIMARY_SEED) & (SEVERITY.budget == "r25")].copy()
rows = []
for mk in MODEL_ORDER:
    for method in METHOD_ORDER:
        g = sev[(sev.model_kind == mk) & (sev.method == method)].set_index("severity")
        rows.append({
            "Model": MODEL_LABEL[mk], "Compressor": METHOD_LABEL[method],
            "Dynamic mild": 100 * g.loc[1, "dynamic_scfr"],
            "Dynamic strong": 100 * g.loc[2, "dynamic_scfr"],
            "Frozen mild": 100 * g.loc[1, "frozen_scfr"],
            "Frozen strong": 100 * g.loc[2, "frozen_scfr"],
            "Delta mild": 100 * g.loc[1, "dynamic_minus_frozen"],
            "Delta strong": 100 * g.loc[2, "dynamic_minus_frozen"],
        })
TABLE5 = pd.DataFrame(rows)
write_csv(TABLE5.round(2), "Table5_seed42.csv")
display(TABLE5.round(2))


## 7. Table 6 — pooled monitor audit

In [ ]:
mon = MONITOR[(MONITOR.seed == PRIMARY_SEED) & (MONITOR.budget == "r25")].copy()
rows = []
for mk in MODEL_ORDER:
    for method in METHOD_ORDER:
        g = mon[(mon.model_kind == mk) & (mon.method == method)].set_index("predictor")
        j = g.loc["support_instability"]
        u = g.loc["clean_uncertainty"]
        rows.append({
            "Model": MODEL_LABEL[mk], "Compressor": METHOD_LABEL[method],
            "Jaccard AUROC": j.raw_auroc,
            "Jaccard CI low": j.raw_ci_low,
            "Jaccard CI high": j.raw_ci_high,
            "U_pm": j.orientation_free_upper_95,
            "Uncertainty AUROC": u.raw_auroc,
            "Uncertainty CI low": u.raw_ci_low,
            "Uncertainty CI high": u.raw_ci_high,
            "L95": u.raw_ci_low,
        })
TABLE6 = pd.DataFrame(rows)
write_csv(TABLE6.round(3), "Table6_seed42.csv")
display(TABLE6.round(3))


## 8. Seed-2026 replication and final integrity checks

In [ ]:
rep_summary = SUMMARY[(SUMMARY.seed == REPLICATION_SEED) & (SUMMARY.budget == "r25")].copy()
rep_monitor = MONITOR[(MONITOR.seed == REPLICATION_SEED) & (MONITOR.budget == "r25")].copy()

REPLICATION_CONTRASTS = rep_summary[[
    "model_name", "model_kind", "method", "dynamic_scfr", "frozen_scfr",
    "dynamic_minus_frozen", "dynamic_minus_frozen_ci_low", "dynamic_minus_frozen_ci_high"
]].copy()
REPLICATION_CONTRASTS["dynamic_scfr"] *= 100
REPLICATION_CONTRASTS["frozen_scfr"] *= 100
REPLICATION_CONTRASTS["dynamic_minus_frozen"] *= 100
REPLICATION_CONTRASTS["dynamic_minus_frozen_ci_low"] *= 100
REPLICATION_CONTRASTS["dynamic_minus_frozen_ci_high"] *= 100
write_csv(REPLICATION_CONTRASTS.round(3), "seed2026_intervention_replication.csv")
write_csv(rep_monitor, "seed2026_monitor_replication.csv")

div = REPLICATION_CONTRASTS[REPLICATION_CONTRASTS.method == "divprune"].dynamic_minus_frozen
apet = REPLICATION_CONTRASTS[REPLICATION_CONTRASTS.method == "apet"].dynamic_minus_frozen
jaccard = rep_monitor[rep_monitor.predictor == "support_instability"].orientation_free_upper_95
unc = rep_monitor[rep_monitor.predictor == "clean_uncertainty"].raw_ci_low

# Mandatory reproducibility checks.
assert len(RUNS) == 6
assert (RUNS.result_rows == EXPECTED_RESULTS_PER_RUN).all()
assert (RUNS.case_rows == EXPECTED_CASES_PER_RUN).all()
assert RUNS.preflight_pass.all()
assert int(RUNS.result_rows.sum()) == 708000
assert (primary.dynamic_minus_frozen > 0).all()
assert (primary.dynamic_minus_frozen_ci_low > 0).all()
assert (mon[mon.predictor == "support_instability"].orientation_free_upper_95 < PRACTICAL_AUROC).all()
assert (mon[mon.predictor == "clean_uncertainty"].raw_ci_low > PRACTICAL_AUROC).all()
assert (rep_summary.dynamic_minus_frozen > 0).all()
assert (rep_summary.dynamic_minus_frozen_ci_low > 0).all()
assert float(jaccard.max()) < PRACTICAL_AUROC
assert float(unc.min()) > PRACTICAL_AUROC
assert int(CELL_COUNTS.iloc[0].eligible_config_min) == 256
assert int(CELL_COUNTS.iloc[0].eligible_config_max) == 857
assert int(CELL_COUNTS.iloc[0].eligible_cluster_min) == 129
assert int(CELL_COUNTS.iloc[0].eligible_cluster_max) == 430

# Explicitly verify the visually surprising Table-5 entry is a rounding coincidence.
ivl = sev[(sev.model_kind == "internvl35") & (sev.method == "divprune")].set_index("severity")
ivl_mild = 100 * float(ivl.loc[1, "dynamic_scfr"])
ivl_strong = 100 * float(ivl.loc[2, "dynamic_scfr"])
assert round(ivl_mild, 2) == 14.24 and round(ivl_strong, 2) == 14.24
assert not np.isclose(ivl_mild, ivl_strong, atol=1e-9, rtol=0)

verification = {
    "pass": True,
    "protocol_tag": PROTOCOL_TAG,
    "completed_runs": 6,
    "total_inference_rows": int(RUNS.result_rows.sum()),
    "total_case_rows": int(RUNS.case_rows.sum()),
    "seed42_all_six_intervention_cis_above_zero": True,
    "seed2026_all_six_intervention_cis_above_zero": True,
    "seed2026_divprune_delta_pp_range": [float(div.min()), float(div.max())],
    "seed2026_apet_delta_pp_range": [float(apet.min()), float(apet.max())],
    "seed2026_max_jaccard_orientation_free_bound": float(jaccard.max()),
    "seed2026_min_uncertainty_lower95": float(unc.min()),
    "table4_eligible_config_range": [256, 857],
    "table4_eligible_cluster_range": [129, 430],
    "internvl_divprune_dynamic_mild_unrounded_pct": ivl_mild,
    "internvl_divprune_dynamic_strong_unrounded_pct": ivl_strong,
}
(META_ROOT / "paper_verification.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")

print(json.dumps(verification, indent=2))
print("\nAll checks passed. Paper-ready outputs are in:", META_ROOT)
